# Pilot runs — slm-audio-evidence (Kaggle)

Notebook **Settings** (правая панель) → **Accelerator: GPU T4 x2** (НЕ P100 -- Pascal, compute capability 6.0, несовместим с preinstalled PyTorch в образе Kaggle; ячейка 4 ниже проверяет это автоматически и падает с понятной ошибкой, если аккселератор не тот) → **Internet: On** (нужен для `git clone`/`pip install`/скачивания весов). Затем — ячейки сверху вниз.

По умолчанию инференс НЕ перезапускается: `responses.jsonl` и ручная разметка `manual-M1` уже закоммичены в `results/`. Ячейки ниже сразу переходят к прогону LLM-судьи и сравнению с ней — аудио для этого не нужно (судья читает только текстовый транскрипт из манифеста).

Нужно пересчитать инференс с нуля (новые аудио/модели/промпты)? Поставь `RUN_INFERENCE = True` в соответствующей ячейке — тогда понадобится `pilot_audio.zip`, прикреплённый как Kaggle Dataset (правая панель → Add Data → Upload), путь ниже (`AUDIO_ZIP_PATH`).

Нет доступа на чтение/клонирование репозитория? Ниже (ячейка 2) есть флаг `USE_LOCAL_ZIP` — поставь `True` и прикрепи zip репозитория как Kaggle Dataset вместо `git clone`.

In [ ]:
!nvidia-smi -L

In [ ]:
import subprocess

def sh(cmd: str) -> None:
    subprocess.run(cmd, shell=True, check=True)

# 2026-07-22: this was pointing at PolinaSh-main/slm-audio-evidence.git, a STALE fork 23 commits
# behind the real working branch (no authorship-rewrite/main-merge/judge-comparison-infra work
# from mid-July onward) -- Kaggle was silently running old src/*.py all session (notebook cell
# edits looked current because those are pasted in directly, not pulled from git). The actual
# up-to-date fork is slm-audio-evidence-judge-fork.git -- confirmed identical to the local
# working tree before this fix. Switch back to REPO_URL of
# https://github.com/ladnlav/slm-audio-evidence.git and REPO_BRANCH="main" once merged upstream.
REPO_URL = "https://github.com/PolinaSh-main/slm-audio-evidence-judge-fork.git"
REPO_BRANCH = "m3/llm-judge"
LOCAL_DIR = "slm-audio-evidence"  # explicit -- git clone otherwise names the folder after the
# repo itself (here "slm-audio-evidence-judge-fork"), breaking the %cd below, which assumes this
# fixed name regardless of which fork/repo REPO_URL points at.

# False (default): git clone from GitHub (needs read access to the repo).
# True: no repo access -- unzip a manually uploaded copy instead (see REPO_ZIP_PATH below).
USE_LOCAL_ZIP = False
REPO_ZIP_PATH = "/kaggle/input/slm-audio-evidence/slm-audio-evidence.zip"

import os

if USE_LOCAL_ZIP:
    sh(f"mkdir -p {LOCAL_DIR} && unzip -q -o {REPO_ZIP_PATH} -d {LOCAL_DIR}")
elif os.path.isdir(LOCAL_DIR):
    # 2026-07-22: a plain `git clone` here fails outright if this cell (or an earlier session in
    # the same warm kernel/Working dir) already checked out LOCAL_DIR once -- "destination path
    # ... already exists". Update in place instead of cloning fresh. `git remote set-url` first
    # (not just `git pull`) so this is also correct if LOCAL_DIR was cloned from a since-changed
    # REPO_URL (e.g. the stale-fork -> real-fork switch earlier this project) -- always ends up
    # tracking whatever REPO_URL/REPO_BRANCH say right now, regardless of prior state. Hard
    # reset, not merge -- this notebook only ever reads the repo, never commits into this clone,
    # so there is nothing local here worth preserving.
    sh(
        f"cd {LOCAL_DIR} && git remote set-url origin {REPO_URL} && "
        f"git fetch origin {REPO_BRANCH} && git checkout {REPO_BRANCH} && "
        f"git reset --hard origin/{REPO_BRANCH}"
    )
else:
    sh(f"git clone --branch {REPO_BRANCH} {REPO_URL} {LOCAL_DIR}")
%cd slm-audio-evidence

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU visible -- Settings (right panel) -> Accelerator, pick GPU T4 x2, "
        "then Restart session and rerun from cell 2."
    )

name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print(f"GPU: {name} (compute capability {capability[0]}.{capability[1]})")
if capability < (7, 0):
    raise RuntimeError(
        f"{name} (compute capability {capability[0]}.{capability[1]}) is too old for the "
        "preinstalled PyTorch build (needs >= 7.0, e.g. T4/V100/A100). Seen in practice: "
        "P100 (Pascal, 6.0) fails here. Settings (right panel) -> Accelerator -> switch to "
        "GPU T4 x2, then Restart session and rerun from cell 2."
    )

In [ ]:
# True only to regenerate responses.jsonl from scratch (new audio/models/prompts).
# False (default): skip audio + inference entirely -- responses.jsonl and the
# manual-M1 human labels are already committed in results/.
RUN_INFERENCE = False

# Only read if RUN_INFERENCE is True. Attach pilot_audio.zip as a Kaggle Dataset
# (Add Data -> Upload) and point this at its path under /kaggle/input/.
AUDIO_ZIP_PATH = "/kaggle/input/pilot-audio/pilot_audio.zip"

In [ ]:
if RUN_INFERENCE:
    sh(f'unzip -q -o {AUDIO_ZIP_PATH} -d .')
    sh('ls data/audio/spoken_squad_test | head -3')
    import json, os
    rows = [json.loads(l) for l in open('data/manifests/pilot.jsonl', encoding='utf-8')]
    miss = [r['id'] for r in rows if not os.path.exists(r['audio_path'])]
    print(len(rows), 'items,', len(miss), 'missing audio')
    print(miss[:5])
else:
    print('RUN_INFERENCE=False -- skipping audio upload/check (responses.jsonl already in results/).')

In [ ]:
# Run 1 (the headline): Qwen2-Audio, plain prompt. First run also downloads the weights (~15-30 min).
if RUN_INFERENCE:
    sh('python -m src.inference --model qwen2audio --strategy plain --data data/manifests/pilot.jsonl --out results/')
else:
    print('RUN_INFERENCE=False -- skipping.')

In [ ]:
# Run 2: Qwen2-Audio, IDK prompt
if RUN_INFERENCE:
    sh('python -m src.inference --model qwen2audio --strategy s1_idk --data data/manifests/pilot.jsonl --out results/')
else:
    print('RUN_INFERENCE=False -- skipping.')

In [ ]:
# IMPORTANT before runs 3-4: free VRAM from Qwen2-Audio -- restart the session/kernel
# (Kaggle: top menu -> Run -> Restart session, or the restart icon), then re-run cells
# 2 (clone/cd), 3 (pip install), 4 (flags) and, if RUN_INFERENCE, cell 5 (unzip) -- then continue here.
if RUN_INFERENCE:
    sh('python -m src.inference --model cascade --strategy plain --data data/manifests/pilot.jsonl --out results/')
else:
    print('RUN_INFERENCE=False -- skipping.')

In [ ]:
# Run 4: cascade, IDK prompt
if RUN_INFERENCE:
    sh('python -m src.inference --model cascade --strategy s1_idk --data data/manifests/pilot.jsonl --out results/')
else:
    print('RUN_INFERENCE=False -- skipping.')

In [ ]:
# Zip everything produced so far (run after EACH finished run -- do not wait for all four)
if RUN_INFERENCE:
    sh("zip -q -r results_runs.zip results -x '*.gitkeep'")
    print('results_runs.zip written -- grab it from the notebook Output tab '
          '(Kaggle persists everything under /kaggle/working) once the session ends, or via the file browser now.')
else:
    print('RUN_INFERENCE=False -- nothing new to zip.')

## LLM-судья (категория B) vs ручная разметка manual-M1
`results/<run_id>/responses.jsonl` уже есть (см. выше). Категория B (label=answer) в них размечена людьми вручную (`results/<run_id>/responses_judged.jsonl`, `judge: "manual-M1"`, committed — см. docs/decisions.md) — это уже готовая истина, повторную слепую разметку делать не нужно.

Судья пишет в **отдельную** папку `results/<run_id>/llm_audit_<версия промпта>/`, а не поверх `responses_judged.jsonl` — тот файл с ручной разметкой нельзя перезаписывать, он невоспроизводим.

Запускаем ОДНИМ Python-процессом (не 4 отдельных вызова) — модель-судья (`Qwen/Qwen3-8B`, int8) грузится один раз и переиспользуется на все 4 прогона: быстрее и без риска VRAM-утечек между процессами, как при повторных инференс-прогонах выше. Если увидите OOM — перезапустите сессию/kernel и заново выполните ячейки клонирования, установки зависимостей и эту.

**ЗАМОРОЖЕННЫЙ рабочий конфиг: `judge_v1.txt`, `enable_thinking=False` — 94% (87/93).** Вся серия экспериментов над промптом и режимом (`judge_v2.txt`, thinking, `judge_v3.txt` ± thinking, `judge_v4.txt` без транскрипта) провалилась — ни один вариант не побил простой no-think `judge_v1.txt`. См. docs/decisions.md 2026-07-15/07-16.

**`judge_v4.txt` (v1 без строки TRANSCRIPT) — проверен на полном датасете, ОТКЛОНЁН.** Гипотеза была: раз транскрипт не упомянут в критерии вердикта дословно, может, он и не нужен — и это сэкономило бы токены на prefill. Оказалось не так: 90% (84/93) против 94% (87/93) — минус 4 пункта. Разбор по элементам: 1 случай починился, но 5 сломались, все на коротких легитимных ответах (например `sq-1122-B1`: "Secondary sources of European Union law are based on the treaties." — верно, но короче gold — без транскрипта в контексте судья резче наказывает такую краткость). Вывод: транскрипт не используется explicitly по тексту рубрики, но реально работает как стабилизирующий контекст — держим его в `judge_v1.txt`.

**Кэш весов судьи между сессиями (docs/decisions.md 2026-07-22 — правка после переполнения Output-квоты)**: `Qwen3-8B` (~16 ГБ) по умолчанию качается в `~/.cache/huggingface`. Кэш лежит в `/root/hf_cache` — **НЕ** в `/kaggle/working` (та папка — это "Output" ноутбука с жёсткой квотой, у нас на аккаунте ~19 ГБ; один Qwen3-8B уже занимает большую часть, а ниже в лестнице кандидатов ещё несколько моделей по 15-55 ГБ каждая — если положить их туда же, квота кончится на первом-втором кандидате). `/root/hf_cache` живёт на отдельном, гораздо большем локальном диске контейнера — не персистится между сессиями сам по себе, но нам и не нужно: важны только результаты (`results/`), не сами веса.

Если есть сохранённый кэш с прошлой сессии (прикреплённый Kaggle Dataset) — ячейка ниже копирует его в `/root/hf_cache` и сразу проверяет размер (`du -sh`) — если там не несколько ГБ, а пара КБ/МБ, копирование не удалось и разбираться нужно ДО прогона судьи, а не после.

**Про `HF_HUB_OFFLINE` (пробовали 2026-07-22, откатили)**: форсировать оффлайн-режим после копирования кэша выглядело логично (кэш ведь уже есть, зачем сеть), но на практике `from_pretrained` и без этого флага уже вёл себя правильно — короткая проверка ревизии на Hub (пара КБ, видна в логе как "Fetching N files" / "Download complete: 0.00/0.00" — это НЕ повторная закачка весов) и переиспользование локальных весов без скачивания. А форсированный оффлайн-режим требует, чтобы кэш был идеально самодостаточен (верные `refs/`-указатели и т.д.) без права на любую сетевую подстраховку — стоило скопированному кэшу оказаться чуть не таким, и вместо тихой доработки по сети получили жёсткий `OSError`. Поэтому флаг убран — пусть эта одна маленькая сетевая проверка остаётся, она не стоит той хрупкости, которую даёт полный оффлайн.

**Кэширование между сессиями (актуальный способ, docs/decisions.md 2026-07-22)**: сырой HF-кэш (папка `hf_cache` с `blobs`+`snapshots`-ссылками) как Kaggle Dataset НЕ работает надёжно — Kaggle плохо сохраняет ссылки внутри файлов при загрузке датасета, и `snapshots/` (без которой нельзя понять, где какой файл) может потеряться. Вместо этого — см. отдельную "одноразовую утилиту" через 2 ячейки ниже: она сохраняет модель как обычные именованные файлы (`model.save_pretrained`/`tokenizer.save_pretrained`), которые Dataset уже не испортит. Инструкция там же.

In [ ]:
import os

# NOT under /kaggle/working -- that's the notebook's persisted "Output" with a hard quota
# (~19 GB on this account); Qwen3-8B alone (~16 GB) nearly filled it, leaving no room for the
# candidate ladder's several other models below (some 15-55 GB each). /root/hf_cache lives on
# the container's separate, much bigger local disk -- doesn't persist across sessions on its
# own, but that's fine: only results/ needs to survive, not the raw weights.
#
# IMPORTANT: HF_HOME is read by huggingface_hub ONCE, at its first import in this process --
# setting it here only works if this is the FIRST time in this kernel that
# `transformers`/`huggingface_hub` gets imported (i.e. right after Run -> Restart session, cells
# 2/3/4 then this one). Re-running just this cell in an already-warm kernel silently ignores the
# new value and keeps using whatever cache dir was resolved at the first import.
HF_CACHE_DIR = "/root/hf_cache"
os.environ["HF_HOME"] = HF_CACHE_DIR  # set, not setdefault -- must win even if this cell reruns

# 2026-07-22: the raw HF hub-cache (blobs + snapshots/ symlinks) does NOT survive being packaged
# as a Kaggle Dataset -- Kaggle's upload path doesn't reliably preserve symlinks, and the
# snapshots/ directory (the ONLY thing that maps blobs to real filenames like config.json) came
# back missing on a real attempt, causing a hard OSError even though the raw bytes were present.
# `pasheviakova/qwen3-cache` was exactly this -- DISABLED below, do not re-attach/re-enable it.
# Replacement: the "flat cache" utility cell right after this one saves the model as plain named
# files (no blobs, no symlinks) instead -- safe to package as a Dataset. Once you've built one,
# set FLAT_MODEL_DIR to its attached path and this block uses it as a local model_id, no download.
_prebuilt_cache = None  # old path was ".../pasheviakova/qwen3-cache/hf_cache" -- BROKEN, left disabled
# Confirmed 2026-07-22: kagglehub.dataset_upload(handle, local_dir) uploads local_dir's CONTENTS
# as the dataset root -- config.json etc. sit directly at the top, no extra subfolder of their
# own. BUT the mount point Kaggle gives it on THIS account is one level deeper than the plain
# "/kaggle/input/<slug>" guess from earlier: confirmed via `!ls -la /kaggle/input/datasets/*/`
# to be "/kaggle/input/datasets/<username>/<slug>/" (an extra "datasets/<username>/" layer).
# If this stops matching after a Kaggle change, re-run that ls to see the real path.
FLAT_MODEL_DIR = "/kaggle/input/datasets/pasheviakova/qwen3-8b-flat-cache"  # None to force a fresh download
if FLAT_MODEL_DIR and os.path.isdir(FLAT_MODEL_DIR):
    sh(f"du -sh {FLAT_MODEL_DIR}")  # sanity check -- should print several GB, not a few KB/MB
    print(f"Using flat model dir {FLAT_MODEL_DIR} -- no HF cache/download involved at all.")
elif _prebuilt_cache and os.path.isdir(_prebuilt_cache) and not os.path.isdir(HF_CACHE_DIR):
    sh(f"cp -r {_prebuilt_cache} {HF_CACHE_DIR}")
    sh(f"du -sh {HF_CACHE_DIR}")  # size alone does NOT prove the cache is usable -- see 2026-07-22 note above
    print(f"Reused cached weights from {_prebuilt_cache} -- no download needed.")
    # NOT forcing HF_HUB_OFFLINE here (tried 2026-07-22, reverted, see docs/decisions.md):
    # from_pretrained's normal behavior -- a cheap metadata check against the Hub, then reusing
    # the local weight blobs without re-downloading them -- already worked fine and cost only a
    # few KB of network traffic ("Fetching N files" / "Download complete: 0.00/0.00" is that
    # check succeeding, not a real download). Forcing pure offline mode requires the cache to be
    # perfectly self-sufficient (correct refs/ pointers etc.) with no fallback if anything about
    # the copied cache is slightly off -- that turned the harmless metadata check into a hard
    # failure in practice. Letting it hit the network for that one small check is the more
    # robust choice.
else:
    print("No usable cache attached -- Qwen3-8B will download fresh from the Hub (~15-20 min).")

from src.judges import build_judge
from src.judges.dev_subset import DEV_SUBSET
from src.run_eval import run_evaluation

RUNS = [
    "qwen2audio_plain_20260712",
    "qwen2audio_s1_idk_20260712",
    "cascade_plain_20260712",
    "cascade_s1_idk_20260712",
]

# judge_v1.txt (FROZEN default: 94% no-think, see docs/decisions.md). judge_v2.txt, judge_v3.txt,
# and judge_v4.txt (v1 without the TRANSCRIPT line -- tested, cost 4 points, rejected) are all
# rejected experiments kept for the record, do not use for real runs. Output dir is named
# after the prompt, so switching this and rerunning does NOT overwrite other results.
JUDGE_PROMPT = "judge_v1.txt"
AUDIT_SUBDIR = "llm_audit_" + JUDGE_PROMPT.replace("judge_", "").replace(".txt", "")  # -> llm_audit_v1

# "dev_subset" (fast, ~23 fixed items -- deliberately weighted toward every item any past config
# has ever disagreed with manual-M1 on, see src/judges/dev_subset.py -- expect a LOWER % here than
# on "full", that's by design, not a regression) / "full" (100% of data, 93 items -- the number
# that actually goes in the PR/report) / "tiered" (fast+slow cascade -- NOT recommended right
# now, see markdown above). Shared with the candidate-ladder loop further down (same variable).
JUDGE_MODE = "dev_subset"

# Only applies to "dev_subset"/"full" ("tiered" manages its own fast/slow toggle internally).
# False = best validated config so far (94% on judge_v1.txt). Thinking did NOT beat no-think on
# this pilot -- only set True when specifically testing whether reasoning room helps a NEW
# prompt (e.g. judge_v3.txt) actually apply its instruction. Change one variable at a time.
JUDGE_THINKING = False

# model_id: the flat local dir if you built/attached one, otherwise fall back to the Hub repo id
# (LocalHFJudge just forwards model_id to from_pretrained either way -- no code change needed).
_model_id_kwargs = {"model_id": FLAT_MODEL_DIR} if FLAT_MODEL_DIR and os.path.isdir(FLAT_MODEL_DIR) else {}

if JUDGE_MODE == "tiered":
    judge_backend, judge_kwargs = "tiered", {**_model_id_kwargs}
else:
    judge_backend, judge_kwargs = "local", {"enable_thinking": JUDGE_THINKING, **_model_id_kwargs}

print(f"Parameters set -- JUDGE_PROMPT={JUDGE_PROMPT}, JUDGE_MODE={JUDGE_MODE}, JUDGE_THINKING={JUDGE_THINKING}. "
      "Run the next cell to build the judge and evaluate (rerun THIS cell first if you change any of the above).")

In [ ]:
# 2026-07-22: split out of the params cell above -- M4 asked for this so changing JUDGE_MODE/
# JUDGE_PROMPT/JUDGE_THINKING and rerunning doesn't also re-touch cache setup/env vars every
# time, and rerunning the actual (slow) build+eval doesn't require re-editing params first.
#
# Rerunning THIS cell in an already-warm kernel (e.g. just to flip JUDGE_MODE or JUDGE_PROMPT in
# the cell above) used to try to load a second copy of the model while the first `judge` from the
# previous run was still alive and holding VRAM -- `judge = build_judge(...)` builds the new
# instance BEFORE rebinding the name, so the old one hadn't been freed yet. With int8 that showed
# up as "Some modules are dispatched on the CPU or the disk" (bitsandbytes refuses partial
# CPU/disk offload), not as a plain CUDA OOM -- easy to misread as caused by the JUDGE_MODE/
# JUDGE_PROMPT change itself. Free the previous instance first so reruns are always safe.
if "judge" in dir():
    import gc
    del judge
    gc.collect()
    torch.cuda.empty_cache()

judge = build_judge(judge_backend, prompt_name=JUDGE_PROMPT, **judge_kwargs)  # Qwen3-8B, int8 -- loads once (~2-4 min)
for run_id in RUNS:
    subset_ids = set(DEV_SUBSET.get(run_id, [])) if JUDGE_MODE == "dev_subset" else None
    run_evaluation(
        manifest_path="data/manifests/pilot.jsonl",
        responses_path=f"results/{run_id}/responses.jsonl",
        out_dir=f"results/{run_id}/{AUDIT_SUBDIR}",  # separate dir -- never touches committed manual-M1
        judge=judge,
        subset_ids=subset_ids,
    )

if hasattr(judge, "escalated_count"):  # only TieredJudge tracks this
    print(f"[tiered] escalated {judge.escalated_count}/{judge.total_count} items to the slow (thinking) pass")

### Одноразовая утилита: сохранить веса судьи как обычные файлы + загрузить как Kaggle Dataset кодом

Не через `/kaggle/working` и не через кнопку "New Dataset" — та привязана к Output-квоте (~19 ГБ), а некоторые модели из лестницы кандидатов (например `Qwen3.6-27B`, ~54 ГБ) в неё не влезут в принципе. Вместо этого — библиотека `kagglehub` (уже предустановлена в Kaggle-ноутбуках, авторизация автоматическая): `kagglehub.dataset_upload(handle, local_dir)` грузит датасет из ЛЮБОЙ папки на диске, лимит — 100 ГБ на датасет, а не 19. Источники: [Kaggle/kagglehub на GitHub](https://github.com/Kaggle/kagglehub), [Kaggle: Create large datasets (up to 100GB) from a notebook](https://www.kaggle.com/getting-started/228867).

Не часть основного потока — запускать РУКАМИ, один раз, ПОСЛЕ того как ячейка выше успешно загрузила модель (переменная `judge` уже существует, модель реально в памяти). Ячейка ниже делает всё сама: сохраняет плоские файлы в `/root/qwen3_flat` (большой диск, без квоты) и сразу грузит их как датасет `<твой_логин>/qwen3-8b-flat-cache` — руками ничего кликать не нужно.

**В следующей сессии**: прикрепи получившийся датасет через **Add Data** (он появится в списке твоих датасетов сразу после загрузки), Kaggle покажет путь монтирования — **проверено на практике 2026-07-22: файлы (`config.json`, `model.safetensors` и т.д.) лежат прямо в корне датасета, без своей отдельной подпапки**. НО сам путь монтирования на этом аккаунте оказался на уровень глубже, чем казалось сначала: не `/kaggle/input/<slug>`, а **`/kaggle/input/datasets/<логин>/<slug>`** (лишний слой `datasets/<логин>/`) — если сомневаешься, проверь командой `!ls -la /kaggle/input/datasets/*/` и сверься с реальным выводом. Подставь получившийся путь в `FLAT_MODEL_DIR` в ячейке судьи выше. Старый `pasheviakova/qwen3-cache` (битый сырой HF-кэш) больше не используем.

**Тот же самый способ — для ОСТАЛЬНЫХ моделей лестницы (Ministral-3-8B, Gemma-4-12B, Qwen3.5-9B, Qwen3.6-27B)**: `model.save_pretrained`/`tokenizer.save_pretrained` всегда пишут одинаковый плоский набор файлов независимо от архитектуры модели, так что тот же паттерн "Add Data -> `/kaggle/input/datasets/<логин>/<slug>` -> в `model_id`" сработает для любой из них один в один. Раньше это было реализовано только для замороженного Qwen3-8B — теперь то же самое встроено прямо в цикл кандидатов (флажок `SAVE_FLAT_CACHE`, ячейка ниже цикла): выставь `True`, и каждая `local`-модель после прогона сама сохранится плоско и загрузится как отдельный датасет `<логин>/<имя-модели>-flat-cache`, без ручного повторения этой ячейки под каждую модель. Ячейка сама печатает точный путь монтирования для каждой загруженной модели — копируй его оттуда, а не собирай вручную.

In [ ]:
# Сохранить уже загруженную модель-судью как обычные файлы (без blobs/symlinks) на большой
# локальный диск -- НЕ /kaggle/working (там 19 ГБ квота, некоторые будущие модели крупнее) --
# и сразу загрузить как Kaggle Dataset кодом через kagglehub (лимит 100 ГБ, авторизация внутри
# ноутбука уже есть, см. markdown выше).
import kagglehub

FLAT_CACHE_DIR = "/root/qwen3_flat"
KAGGLE_USERNAME = "pasheviakova"  # поправь, если логин другой
DATASET_HANDLE = f"{KAGGLE_USERNAME}/qwen3-8b-flat-cache"

_backend = judge.judge_backend if hasattr(judge, "judge_backend") else judge  # unwrap TieredJudge, if any
_backend.model.save_pretrained(FLAT_CACHE_DIR)
_backend.tokenizer.save_pretrained(FLAT_CACHE_DIR)
sh(f"du -sh {FLAT_CACHE_DIR}")  # должно быть ~16 ГБ, как и сама модель -- иначе запись не удалась

kagglehub.dataset_upload(DATASET_HANDLE, FLAT_CACHE_DIR)
# Проверено 2026-07-22 на этом аккаунте: путь монтирования на уровень глубже, чем можно
# подумать -- "/kaggle/input/datasets/<логин>/<slug>", а не "/kaggle/input/<slug>". Печатаем
# точный ожидаемый путь, чтобы в след. сессии просто скопировать его в FLAT_MODEL_DIR (и на
# всякий случай перепроверить командой !ls -la /kaggle/input/datasets/*/).
expected_mount = f"/kaggle/input/datasets/{KAGGLE_USERNAME}/qwen3-8b-flat-cache"
print(f"Готово: {DATASET_HANDLE} загружен из {FLAT_CACHE_DIR}. "
      f"В следующей сессии прикрепи его через Add Data и подставь "
      f"FLAT_MODEL_DIR = \"{expected_mount}\" в ячейке судьи выше "
      f"(сверься с !ls -la /kaggle/input/datasets/*/).")

Сравнить свежие вердикты судьи с manual-M1 (гейт согласия ROLE_M3: 80%; если ниже — не публиковать LLM-judge цифры, см. вывод команды). Использует тот же `AUDIT_SUBDIR` и отдельный `--out` на промпт, что и ячейка выше — сравнения `judge_v1` и `judge_v2` не затирают друг друга:

In [ ]:
sh(
    'python scripts/audit_judge.py compare '
    '--judged "results/*/responses_judged.jsonl" '
    f'--judge-subdir {AUDIT_SUBDIR} '
    f'--out results/judge_audit_{AUDIT_SUBDIR.replace("llm_audit_", "")}'
)

In [ ]:
# Собрать вердикты судьи + отчёт о согласии в zip (оба промпта, если гоняли оба)
!zip -q -r judge_audit.zip results/*/llm_audit_* results/judge_audit_* -x '*.gitkeep'
print('judge_audit.zip written -- grab it from the notebook Output tab '
      '(Kaggle persists everything under /kaggle/working).')

## Лестница судей-кандидатов (ROLE_M4 task 1, docs/decisions.md 2026-07-21)

Тестируем ДОПОЛНИТЕЛЬНЫЕ модели-судьи поверх уже замороженного `Qwen/Qwen3-8B` (`judge_v1.txt`, no-think, 94% — ячейка выше, **не трогаем**). **2026-07-22: каждая модель теперь в своей ОТДЕЛЬНОЙ ячейке** (было — один большой цикл по списку `CANDIDATES`) — можно запускать/перезапускать любую модель независимо от остальных, в любом порядке, пропускать те, что пока не нужны (например VLM-кандидатов до проверки класса загрузки), и повторить один упавший прогон (например Gemini после квоты), не трогая уже готовые. Общие настройки и функция `run_candidate()` — в первой ячейке ниже (выполнить один раз), дальше — по одной ячейке на модель.

Каждый кандидат пишет вердикты в СВОЮ папку `results/<run_id>/llm_audit_<judge.name>/` — уже размеченный `responses_judged.jsonl` (ручная разметка manual-M1) не трогается ни при каком кандидате. `run_candidate()` сама выгружает GPU-модель (`del` + `torch.cuda.empty_cache()`) после каждого вызова, иначе OOM на 16 ГБ (та же проблема, что и в ячейке 13 выше) — при последовательном запуске ячеек это работает так же, как раньше в цикле.

Требует, чтобы ячейки 2, 3, 4 и 13 выше уже были выполнены — переиспользует `sh()`, `RUNS`, `JUDGE_MODE`, `DEV_SUBSET` оттуда.

**`model_id` проверены напрямую на huggingface.co 2026-07-21** (не плейсхолдеры):
- `mistralai/Ministral-3-8B-Instruct-2512-BF16` — не "Mistral-3", а **Ministral-3** (линейка Mistral для edge/small моделей); Apache 2.0. У Instruct-варианта thinking-переключателя НЕТ вообще. Reasoning — **отдельный чекпоинт** `mistralai/Ministral-3-8B-Reasoning-2512`, тоже без переключателя, но всегда рассуждает (как DeepSeek-R1-Distill — дешёвого no-think режима для него не существует).
- `Qwen/Qwen3.5-9B` и `Qwen/Qwen3.6-27B` (не `-Instruct`, это сами чат-модели, как и наш `Qwen3-8B`) — оба Apache 2.0, оба подтверждают `enable_thinking` в `apply_chat_template` (тот же интерфейс, что уже реализован в `LocalHFJudge`).

**Важная находка при проверке**: `Ministral-3-8B`, `Qwen3.5-9B` и `Qwen3.6-27B` — все три оказались vision-language моделями (multimodal, с vision-энкодером), не чисто текстовыми. `LocalHFJudge` (`src/judges/local_hf.py`) грузит через `AutoModelForCausalLM` — не проверено, подхватит ли этот класс архитектуру VLM-чекпоинта корректно в text-only режиме (может потребоваться другой Auto-класс, например специфичный для image-text-to-text). Рекомендация: запусти ячейку каждой такой модели отдельно (они уже в своих ячейках ниже) и убедись, что ошибки класса нет — если есть, это реальный пробел в `LocalHFJudge`, а не опечатка в `model_id`.

Для Gemma-4 (`google/gemma-4-12b-it`, id не менялся) отдельно проверь в её `config.json`/model card, действительно ли `enable_thinking` переключается в рантайме, или reasoning встроен всегда (источники противоречат друг другу, см. обсуждение 2026-07-21) — если встроен всегда, `enable_thinking=False` ничего не изменит и это НЕ баг, а свойство модели.

Gemini не грузит веса — вместо GPU нужен `GEMINI_API_KEY`. Положи его в Kaggle Secrets (правая панель → Add-ons → Secrets), а не в код ячейки — следующая ячейка подхватит его оттуда автоматически. Модель зафиксирована на `gemini-2.5-flash-lite` (docs/decisions.md 2026-07-22) — единственная free-tier модель Gemini, чей дневной лимит (1000 запросов) реально покрывает прогон на весь пилот; `gemini-2.5-flash` упирается в квоту (250/день) на середине одного прогона. Не подменяем модель на лету при исчерпании квоты (осознанно отклонённая идея — вердикты от разных judge-моделей внутри одного датасета несравнимы), только пауза между вызовами + честное ожидание, если лимит всё же задет.

In [ ]:
import os

# Kaggle Secrets -> GEMINI_API_KEY, only if the "gemini" candidate below is actually used.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault("GEMINI_API_KEY", UserSecretsClient().get_secret("GEMINI_API_KEY"))
except Exception:
    pass  # no Secret attached -- GeminiJudge raises a clear RuntimeError if gemini is selected without it

import gc
import json
import re
import time
from pathlib import Path

import torch
from huggingface_hub import scan_cache_dir

from src.judges import build_judge
from src.run_eval import run_evaluation

# 2026-07-22: shared setup + run_candidate() for the candidate ladder below -- run this ONE cell
# once, then each model has its own cell further down (see markdown above for why: independent
# reruns, any order, skip/redo one without touching the rest).
MANIFEST_PATH = "results/judge_audit_multi/run_manifest.json"

# Off by default: roughly doubles disk I/O per candidate (save flat copy + upload, on top of the
# download) and adds real time (uploading tens of GB per model) -- only turn on for a candidate
# you actually want to reuse across sessions without redownloading (see cells 14/15 above for the
# pattern this reuses).
SAVE_FLAT_CACHE = False
KAGGLE_USERNAME = "pasheviakova"  # поправь, если логин другой


def _slugify(name: str, max_len: int = 35) -> str:
    """Kaggle dataset handles: lowercase letters/digits/hyphens only, and there's a length cap
    on the slug (not 100% confirmed exact number -- 35 is a conservative guess so
    "<slug>-flat-cache" stays comfortably short). Truncating naively from the right can eat the
    "-v1"/"-v2" suffix on long names (e.g. Ministral's) and make two different candidates collide
    on the same dataset handle -- so the suffix is carved out first and always preserved,
    only the model-name part in the middle gets shortened.
    """
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    m = re.search(r"-(v\d+)$", slug)
    suffix = f"-{m.group(1)}" if m else ""
    base = slug[: -len(suffix)] if suffix else slug
    return base[: max_len - len(suffix)].rstrip("-") + suffix


def save_flat_and_upload(cand_judge) -> None:
    """Same save_pretrained + kagglehub.dataset_upload dance as the frozen-judge utility cell,
    generalized to any local candidate -- see docs/decisions.md 2026-07-22 (raw HF cache loses
    its snapshots/ symlinks when packaged as a Kaggle Dataset; flat named files don't have that
    problem). Writes to /root, not /kaggle/working -- some candidates here are tens of GB,
    bigger than the ~19 GB Output quota entirely.
    """
    import kagglehub

    slug = _slugify(cand_judge.name)
    flat_dir = f"/root/{slug}_flat"
    handle = f"{KAGGLE_USERNAME}/{slug}-flat-cache"

    cand_judge.model.save_pretrained(flat_dir)
    cand_judge.tokenizer.save_pretrained(flat_dir)
    sh(f"du -sh {flat_dir}")  # sanity check -- should be several GB, not KB/MB

    kagglehub.dataset_upload(handle, flat_dir)
    # Confirmed 2026-07-22 on this account: the mount point is one level deeper than the dataset
    # slug alone -- "/kaggle/input/datasets/<username>/<slug>/", not "/kaggle/input/<slug>/".
    # Print the exact expected path so next session you copy it instead of re-deriving it (and
    # re-check with `!ls -la /kaggle/input/datasets/*/` if Kaggle ever changes this layout).
    expected_mount = f"/kaggle/input/datasets/{KAGGLE_USERNAME}/{slug}-flat-cache"
    print(f"  uploaded {handle} from {flat_dir} -- attach via Add Data next session, "
          f"then set FLAT_MODEL_DIR = \"{expected_mount}\" (verify with !ls -la /kaggle/input/datasets/*/).")


def free_disk_cache_for(model_id: str) -> None:
    """Delete this model's downloaded weights from HF_HOME (cell 13's /root/hf_cache) once
    we're done with it. Even off /kaggle/working, the container's local disk is finite --
    several of these candidates are tens of GB each (Qwen3.6-27B alone is ~54 GB bf16),
    holding all of them on disk at once would still run out of room. Only the raw weight
    cache is freed -- results/ (the actual judge output) is untouched.
    """
    cache_info = scan_cache_dir(os.environ["HF_HOME"])
    revisions = [rev.commit_hash for repo in cache_info.repos if repo.repo_id == model_id for rev in repo.revisions]
    if revisions:
        cache_info.delete_revisions(*revisions).execute()
        print(f"  freed disk cache for {model_id}")


def _append_manifest_entry(entry: dict) -> None:
    """Each candidate cell calls this once, instead of the old single loop building one list in
    memory and writing it at the very end -- with independent per-model cells there's no single
    "end" anymore, and this way results/judge_audit_multi/run_manifest.json is always up to date
    with whatever's actually been run so far, survives a crash in a later cell, and a rerun of
    one model's cell just replaces its own entry (matched by judge_name) instead of duplicating it.
    """
    path = Path(MANIFEST_PATH)
    path.parent.mkdir(parents=True, exist_ok=True)
    entries = json.loads(path.read_text(encoding="utf-8")) if path.exists() else []
    entries = [e for e in entries if e["judge_name"] != entry["judge_name"]]
    entries.append(entry)
    path.write_text(json.dumps(entries, ensure_ascii=False, indent=2), encoding="utf-8")


def run_candidate(backend: str, kwargs: dict, thinking: bool, free_disk_cache: bool = True) -> None:
    """Run one judge candidate across all RUNS and record it in the shared manifest. Pass
    free_disk_cache=False if the very next cell you're about to run reuses the same model_id
    (e.g. Qwen3.5-9B no-think then thinking below) -- skips a pointless redownload.
    """
    cand_judge = build_judge(backend, prompt_name="judge_v1.txt", **kwargs)
    print(f"=== {cand_judge.name} ===")
    t0 = time.time()
    for run_id in RUNS:
        subset_ids = set(DEV_SUBSET.get(run_id, [])) if JUDGE_MODE == "dev_subset" else None
        run_evaluation(
            manifest_path="data/manifests/pilot.jsonl",
            responses_path=f"results/{run_id}/responses.jsonl",
            out_dir=f"results/{run_id}/llm_audit_{cand_judge.name}",  # own dir per candidate
            judge=cand_judge,
            subset_ids=subset_ids,
        )
    elapsed = time.time() - t0
    print(f"=== {cand_judge.name} done in {elapsed:.0f}s ===")
    _append_manifest_entry({
        "judge_name": cand_judge.name,
        "model_id": kwargs.get("model_id", backend),
        "backend": backend,
        "thinking": thinking,
        "seconds": elapsed,
    })
    if backend == "local":
        if SAVE_FLAT_CACHE:
            save_flat_and_upload(cand_judge)
        model_id = kwargs["model_id"]
        del cand_judge
        gc.collect()  # free VRAM before the next candidate -- see markdown above
        torch.cuda.empty_cache()
        if free_disk_cache:
            free_disk_cache_for(model_id)
    else:
        del cand_judge

print("Setup done -- run any of the per-model cells below, in any order.")

**Gemini 2.5 Flash-Lite** — не грузит GPU, нужен `GEMINI_API_KEY` в Kaggle Secrets (см. markdown выше). Проактивно троттлится под free-tier RPM внутри `GeminiJudge` — обычно не должно требовать ретраев вовсе.

In [ ]:
run_candidate("gemini", {"model_id": "gemini-2.5-flash-lite"}, thinking=False)

**Ministral-3-8B Instruct** — VLM checkpoint (see markdown above), no thinking toggle at all. First real attempt at this class -- watch for a class-mismatch error from `AutoModelForCausalLM`.

In [ ]:
run_candidate("local", {"model_id": "mistralai/Ministral-3-8B-Instruct-2512-BF16", "enable_thinking": False}, thinking=False)

**Ministral-3-8B Reasoning** — separate always-on checkpoint (no toggle, always reasons, like DeepSeek-R1-Distill). Also a VLM checkpoint -- same class-mismatch caveat as above.

In [ ]:
run_candidate("local", {"model_id": "mistralai/Ministral-3-8B-Reasoning-2512", "enable_thinking": True}, thinking=True)

**Gemma-4-12B-it** — check its `config.json`/model card for whether `enable_thinking` actually toggles anything at runtime, or reasoning is always-on (see markdown above, sources disagreed as of 2026-07-21); if always-on, `enable_thinking=False` doing nothing is a model property, not a bug here.

In [ ]:
run_candidate("local", {"model_id": "google/gemma-4-12b-it", "enable_thinking": False}, thinking=False)

**Qwen3.5-9B, no-think then thinking** — same `model_id` in both of the next two cells, so the first one passes `free_disk_cache=False` to skip re-downloading for the second. Also a VLM checkpoint (see markdown above) -- watch for the same class-mismatch caveat.

In [ ]:
run_candidate("local", {"model_id": "Qwen/Qwen3.5-9B", "enable_thinking": False}, thinking=False, free_disk_cache=False)

In [ ]:
run_candidate("local", {"model_id": "Qwen/Qwen3.5-9B", "enable_thinking": True}, thinking=True)

**Qwen3.6-27B** — largest candidate (~54 GB bf16); also a VLM checkpoint, same class-mismatch caveat. Consider testing this one in isolation first given its size before running the rest of the ladder in the same session.

In [ ]:
run_candidate("local", {"model_id": "Qwen/Qwen3.6-27B", "enable_thinking": False}, thinking=False)

In [ ]:
# Optional: see which candidates have actually run so far (useful after running only a subset
# of the cells above) before generating the comparison table below.
print(Path(MANIFEST_PATH).read_text(encoding="utf-8") if Path(MANIFEST_PATH).exists()
      else f"{MANIFEST_PATH} does not exist yet -- run at least one candidate cell above first.")

Собрать ОДНУ сравнительную таблицу по всем кандидатам (agreement vs manual-M1, сортировка по убыванию) + разложить провалы каждой модели и, если был thinking, ПОЛНЫЕ reasoning-трейсы по каждому вопросу — в ОТДЕЛЬНЫЕ папки, не в общий отчёт (см. docstring скрипта):

In [ ]:
sh(
    'python scripts/audit_multi_judge.py '
    '--manifest results/judge_audit_multi/run_manifest.json '
    '--judged "results/*/responses_judged.jsonl" '
    '--data-manifest data/manifests/pilot.jsonl '
    '--out results/judge_audit_multi'
)  # prints the comparison table itself -- no need to re-read/re-print it here

In [ ]:
# Собрать в zip: сравнительную таблицу, провалы каждой модели, и (отдельно) reasoning-трейсы
!zip -q -r judge_audit_multi.zip results/judge_audit_multi results/*/llm_audit_llm-* -x '*.gitkeep'
print('judge_audit_multi.zip written -- grab it from the notebook Output tab '
      '(Kaggle persists everything under /kaggle/working).')

### Точечный эксперимент: `judge_v5.txt` (без транскрипта) + thinking, только Apollo 13
Не часть основного цикла выше — маленькая ad-hoc проверка одной гипотезы на одном вопросе (`sq-5207-B2`), чтобы не платить за прогон всего `dev_subset` ради неё. Использует тот же `run_evaluation(..., subset_ids=...)`, что и `JUDGE_MODE="dev_subset"` внутри, но с явным множеством из одного id.

In [ ]:
# Ad-hoc, не часть JUDGE_MODE выше: judge_v5.txt (judge_v3.txt без строки TRANSCRIPT) +
# thinking, только на sq-5207-B2 (Apollo 13). Переиспользует УЖЕ ЗАГРУЖЕННУЮ модель из
# ячейки судьи выше (не build_judge(...) заново) -- вторая копия Qwen3-8B в VRAM упала с
# CUDA OOM на практике (14.56 ГиБ GPU, обе копии не влезли), см. docs/decisions.md 2026-07-16.
adhoc_backend = judge.judge_backend if hasattr(judge, "judge_backend") else judge  # unwrap TieredJudge, if any
adhoc_backend.set_prompt("judge_v5.txt")
adhoc_backend.set_thinking(True)  # also fixes .name (-v2 suffix) so the cache key stays correct
for run_id in RUNS:
    run_evaluation(
        manifest_path="data/manifests/pilot.jsonl",
        responses_path=f"results/{run_id}/responses.jsonl",
        out_dir=f"results/{run_id}/llm_audit_v5_adhoc_apollo13",
        judge=adhoc_backend,
        subset_ids={"sq-5207-B2"},  # only this one -- most runs will just skip it (not judge-routed)
    )